In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Factorized GD on Covariance (Implicit Regularization)\n",
        "\n",
        "This notebook tests the project pipeline:\n",
        "- download prices → returns\n",
        "- compute sample covariance\n",
        "- define operator A(X) (masked entries)\n",
        "- run factorized GD on X = UUᵀ (tiny init vs big init)\n",
        "- compare trace (PSD nuclear norm), spectra, and portfolio backtest\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 0) Setup\n",
        "Make sure your repo structure is:\n",
        "\n",
        "```\n",
        "implicit-cov-gd/\n",
        "  src/\n",
        "  notebooks/\n",
        "```\n",
        "\n",
        "and you run this notebook from the project root (or adjust paths).\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import os, sys\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "# Ensure project root is on Python path\n",
        "PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), \"..\")) if os.getcwd().endswith(\"notebooks\") else os.getcwd()\n",
        "if PROJECT_ROOT not in sys.path:\n",
        "    sys.path.insert(0, PROJECT_ROOT)\n",
        "\n",
        "print(\"Project root:\", PROJECT_ROOT)\n",
        "print(\"Python path OK\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1) Import your project modules"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from src.config import Config\n",
        "from src.etl.utils import download_prices_yfinance, log_returns, train_test_split\n",
        "from src.operators.entry_mask import EntryMaskOperator\n",
        "from src.algorithms.optimize import FactorizedGD\n",
        "from src.models.covariance import SampleCovariance, FactorizedGDCovariance\n",
        "from src.evaluation.backtest import backtest\n",
        "from src.algorithms.optim_utils import eigenvalues_desc, singular_values_desc\n",
        "\n",
        "cfg = Config()\n",
        "cfg"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2) Download data and compute returns\n",
        "You can change tickers/dates in `src/config.py`."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "prices = download_prices_yfinance(cfg.tickers, cfg.start, cfg.end)\n",
        "returns = log_returns(prices)\n",
        "R_train_df, R_test_df = train_test_split(returns, cfg.test_size)\n",
        "\n",
        "print(\"Prices shape:\", prices.shape)\n",
        "print(\"Returns shape:\", returns.shape)\n",
        "print(\"Train shape:\", R_train_df.shape)\n",
        "print(\"Test shape:\", R_test_df.shape)\n",
        "\n",
        "returns.head()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3) Build the operator A(X)\n",
        "We observe only a fraction of covariance entries (underdetermined).\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "n = len(cfg.tickers)\n",
        "A = EntryMaskOperator.random(n=n, frac=cfg.mask_frac, seed=cfg.seed, include_diag=cfg.include_diag)\n",
        "print(\"Number of observed entries (m):\", len(A.omega_upper))\n",
        "print(\"Example Omega indices:\", A.omega_upper[:10])"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4) Baseline: sample covariance"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "Sigma_sample = SampleCovariance().fit(R_train_df.values).covariance_\n",
        "print(\"Sigma_sample shape:\", Sigma_sample.shape)\n",
        "print(\"trace(Sigma_sample):\", float(np.trace(Sigma_sample)))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5) Factorized GD covariance: tiny init vs big init\n",
        "This is your core experiment for implicit regularization.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "gd_tiny = FactorizedGD(\n",
        "    lr=cfg.lr,\n",
        "    n_steps=cfg.n_steps,\n",
        "    init_scale=cfg.init_scale_tiny,\n",
        "    seed=cfg.seed,\n",
        "    log_every=cfg.log_every\n",
        ")\n",
        "gd_big = FactorizedGD(\n",
        "    lr=cfg.lr,\n",
        "    n_steps=cfg.n_steps,\n",
        "    init_scale=cfg.init_scale_big,\n",
        "    seed=cfg.seed,\n",
        "    log_every=cfg.log_every\n",
        ")\n",
        "\n",
        "est_tiny = FactorizedGDCovariance(A, gd_tiny).fit(R_train_df.values)\n",
        "est_big  = FactorizedGDCovariance(A, gd_big).fit(R_train_df.values)\n",
        "\n",
        "Sigma_tiny = est_tiny.covariance_\n",
        "Sigma_big  = est_big.covariance_\n",
        "\n",
        "print(\"trace(Sigma_tiny):\", float(np.trace(Sigma_tiny)))\n",
        "print(\"trace(Sigma_big): \", float(np.trace(Sigma_big)))\n",
        "\n",
        "# Check fit on observed entries\n",
        "y_true = A.forward(Sigma_sample)\n",
        "res_tiny = np.linalg.norm(A.forward(Sigma_tiny) - y_true)\n",
        "res_big  = np.linalg.norm(A.forward(Sigma_big) - y_true)\n",
        "\n",
        "print(\"Residual on observed entries (tiny init):\", res_tiny)\n",
        "print(\"Residual on observed entries (big init): \", res_big)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6) Plot implicit regularization signal: trace(X)\n",
        "For PSD matrices, nuclear norm = trace.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "h_tiny = est_tiny.result.history\n",
        "h_big  = est_big.result.history\n",
        "\n",
        "plt.figure(figsize=(7,4))\n",
        "plt.plot(h_tiny.steps, h_tiny.trace, label=\"tiny init\")\n",
        "plt.plot(h_big.steps,  h_big.trace,  label=\"big init\")\n",
        "plt.xlabel(\"GD steps\")\n",
        "plt.ylabel(\"trace(X) = ||X||_* (PSD)\")\n",
        "plt.title(\"Implicit regularization signal\")\n",
        "plt.legend()\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 7) Spectral plots: eigenvalues of X and singular values of U\n",
        "- eigenvalues of X tell you the covariance factor strength\n",
        "- singular values of U relate to eigenvalues of X (λ(X) = σ(U)²)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "U_tiny = est_tiny.result.U\n",
        "U_big  = est_big.result.U\n",
        "\n",
        "eig_sample = eigenvalues_desc(Sigma_sample)\n",
        "eig_tiny   = eigenvalues_desc(Sigma_tiny)\n",
        "eig_big    = eigenvalues_desc(Sigma_big)\n",
        "\n",
        "plt.figure(figsize=(7,4))\n",
        "plt.semilogy(eig_sample, marker=\"o\", label=\"Sample covariance\")\n",
        "plt.semilogy(eig_tiny,   marker=\"o\", label=\"Recovered (tiny init)\")\n",
        "plt.semilogy(eig_big,    marker=\"o\", label=\"Recovered (big init)\")\n",
        "plt.title(\"Eigenvalue spectra of covariance X\")\n",
        "plt.xlabel(\"Index\")\n",
        "plt.ylabel(\"Eigenvalue (log)\")\n",
        "plt.legend()\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "sv_tiny = singular_values_desc(U_tiny)\n",
        "sv_big  = singular_values_desc(U_big)\n",
        "\n",
        "plt.figure(figsize=(7,4))\n",
        "plt.semilogy(sv_tiny, marker=\"o\", label=\"U (tiny init)\")\n",
        "plt.semilogy(sv_big,  marker=\"o\", label=\"U (big init)\")\n",
        "plt.title(\"Singular values of U\")\n",
        "plt.xlabel(\"Index\")\n",
        "plt.ylabel(\"Singular value (log)\")\n",
        "plt.legend()\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 8) Portfolio backtest (min-variance)\n",
        "We compute weights from Σ_hat (train) and evaluate on test returns.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "R_test = R_test_df.values\n",
        "\n",
        "res_sample = backtest(R_test, Sigma_sample, cfg.ridge)\n",
        "res_tiny   = backtest(R_test, Sigma_tiny, cfg.ridge)\n",
        "res_big    = backtest(R_test, Sigma_big, cfg.ridge)\n",
        "\n",
        "pd.DataFrame([\n",
        "    {\"Method\": \"Sample\", **{k:v for k,v in res_sample.items() if k!='weights'}},\n",
        "    {\"Method\": \"Tiny init\", **{k:v for k,v in res_tiny.items() if k!='weights'}},\n",
        "    {\"Method\": \"Big init\", **{k:v for k,v in res_big.items() if k!='weights'}},\n",
        "])"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 9) Plot cumulative returns (optional but nice for report)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "def cum_wealth(r):\n",
        "    return np.cumprod(1.0 + r)\n",
        "\n",
        "from src.models.portfolio import MinVariancePortfolio\n",
        "\n",
        "r_sample = MinVariancePortfolio(cfg.ridge).fit(Sigma_sample).returns(R_test)\n",
        "r_tiny   = MinVariancePortfolio(cfg.ridge).fit(Sigma_tiny).returns(R_test)\n",
        "r_big    = MinVariancePortfolio(cfg.ridge).fit(Sigma_big).returns(R_test)\n",
        "\n",
        "plt.figure(figsize=(7,4))\n",
        "plt.plot(cum_wealth(r_sample), label=\"Sample\")\n",
        "plt.plot(cum_wealth(r_tiny),   label=\"Tiny init\")\n",
        "plt.plot(cum_wealth(r_big),    label=\"Big init\")\n",
        "plt.title(\"Cumulative wealth (test period)\")\n",
        "plt.xlabel(\"Time\")\n",
        "plt.ylabel(\"Wealth\")\n",
        "plt.legend()\n",
        "plt.tight_layout()\n",
        "plt.show()"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.11"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
